In [ ]:
!nvidia-smi

Tue Feb 24 09:23:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P0             27W /   70W |    7721MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive

# 强制挂载 Google Drive 到 /content/drive 节点
# force_remount=True 可以防止之前挂载的残留导致异常
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16, # 可写可不写；你之前看到默认就是 bf16
    device_map="auto",
)

print("vocab_size:", model.config.vocab_size)
print("pad_token:", tokenizer.pad_token)
print("param dtype:", next(model.parameters()).dtype)


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


vocab_size: 151936
pad_token: <|endoftext|>
param dtype: torch.bfloat16


In [ ]:
from datasets import Dataset

raw_data = [
    {"input": "what a fun party", "output": "🥳🎉"},
    {"input": "good morning sunshine", "output": "☀️😎"},
    {"input": "so tired today", "output": "😴🥱"},
    {"input": "deadline is tomorrow", "output": "😱🆘"},
    {"input": "i love coding", "output": "💻❤️"},
    {"input": "let's go to the gym", "output": "💪🏋️"}
]

raw_dataset = Dataset.from_list(raw_data)

def format_prompt(example):
    return {
        "text": f"User: Translate to emoji: {example['input']}\nAssistant: {example['output']}{tokenizer.eos_token}"
    }

formatted_dataset = raw_dataset.map(format_prompt)

def tokenize_fn(example):
    tokenized = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length", # ✅ 打开 padding
        max_length=64, # emoji 任务很短，64 足够
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_dataset = formatted_dataset.map(
    tokenize_fn,
    remove_columns=formatted_dataset.column_names
)

from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

training_args = TrainingArguments(
    output_dir="./qwen3-emoji",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-5,
    fp16=False,
    bf16=True, #
    logging_steps=10,
    save_steps=500,
    optim="adamw_torch",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,  # 你的数据
    data_collator=data_collator,
)

print("--- 开始训练 ---")
trainer.train()
print("--- 训练完成 ---")


--- 开始训练 ---


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

--- 训练完成 ---


In [ ]:
def test_model(text):
    prompt = (
        "User: Translate to emoji: " + text + "\n"
        "Assistant: "
    )

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=64,           # 翻译 Emoji 64 个 token 顶天了
        do_sample=True,
        temperature=0.6,             # 推理模式推荐 0.6
        top_p=0.95,          # 配合采样，增加多样性
        repetition_penalty=1.2,      # 【核心】增加重复惩罚，防止死循环
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
        # 【关键】把 Assistant: 也加入停止词列表
        stop_strings=["</s>", "\n", "User:", "Assistant:", "<|im_end|>"],
        tokenizer=tokenizer
    )

    print(tokenizer.decode(outputs[0], skip_special_tokens=False))


In [ ]:
# def test_model(text):
#     prompt = f"User: Translate to emoji: {text}\nAssistant: "
#     inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
#     outputs = model.generate(**inputs, max_new_tokens=10)
#     print(tokenizer.decode(outputs[0], skip_special_tokens=True))

test_model("working late at night")

User: Translate to emoji: working late at night
Assistant: 󰀁⚡🌙
    



In [ ]:
# for lora only

# # 将补丁缝合回原模型
# merged_model = model.merge_and_unload()

# # 保存这个完整的模型
# merged_model.save_pretrained("./gemma-3-emoji-full")
# tokenizer.save_pretrained("./gemma-3-emoji-full")

# print("模型已缝合，准备进入压缩环节。")

In [ ]:
trainer.save_model("./qwen3-emoji-full")
tokenizer.save_pretrained("./qwen3-emoji-full")


In [ ]:
!nvidia-smi

In [ ]:
# # 释放 GPU
# del trainer
# del model
# del data_collator

# import torch
# torch.cuda.empty_cache()

# import gc
# gc.collect()

# !nvidia-smi

# print("GPU 已释放，可以继续下一步。")


In [ ]:
import os

# 定义你的专属工作目录，例如 gemma3_finetune
gdrive_work_dir = '/content/drive/MyDrive/colab/2026/qwen3_finetune'

# 1. 先在 Python 中把变量设置到系统环境变量里
os.environ['GDRIVE_WORK_DIR'] = gdrive_work_dir

print(f"✅ 环境变量已注入: {os.environ['GDRIVE_WORK_DIR']}")

# 如果目录不存在，使用 os 模块自动创建
os.makedirs(gdrive_work_dir, exist_ok=True)

print(f"工作目录已准备就绪: {gdrive_work_dir}")

# 可以用 bash 命令验证目录创建成功
!ls -ld {gdrive_work_dir}

print(f"工作目录已准备就绪: {gdrive_work_dir}")
!ls -lR {gdrive_work_dir}

✅ 环境变量已注入: /content/drive/MyDrive/colab/2026/qwen3_finetune
工作目录已准备就绪: /content/drive/MyDrive/colab/2026/qwen3_finetune
drwx------ 2 root root 4096 Feb 20 10:30 /content/drive/MyDrive/colab/2026/qwen3_finetune
工作目录已准备就绪: /content/drive/MyDrive/colab/2026/qwen3_finetune
/content/drive/MyDrive/colab/2026/qwen3_finetune:
total 2365320
-rw------- 1 root root  428520426 Feb 20 13:02 llama.cpp.tar.gz
-rw------- 1 root root 1509347072 Feb 20 13:16 qwen3-0.6b-bf16.gguf
-rw------- 1 root root  484219648 Feb 20 13:16 qwen3-0.6b-q4.gguf


In [ ]:
%%bash
set -e

# 1. 明确定义变量并使用双引号包裹，防止路径解析失败
# 注意：{gdrive_work_dir} 是 Colab 提供的 Python 变量插值
GDRIVE_PATH="${GDRIVE_WORK_DIR}/llama.cpp.tar.gz"
LOCAL_PATH="/content/llama.cpp.tar.gz"

echo "🔎 正在检查云端路径: $GDRIVE_PATH"

# 2. 这里的引号是重中之重！
if [ -f "$GDRIVE_PATH" ]; then
    echo "✅ 成功找到备份包，启动高速拷贝..."

    # 拷贝到 /content/ 以获得更高的磁盘 IO 性能
    cp "$GDRIVE_PATH" "$LOCAL_PATH"

    echo "📦 正在解压编译好的环境到根目录..."
    # -C / 配合 .tar.gz 内部的完整路径结构
    tar -xzf "$LOCAL_PATH" -C /

    echo "✨ 环境恢复完成！验证 bin 目录:"
    ls -F /content/llama.cpp/build/bin/llama-server
else
    echo "❌ 依然找不到文件！"
    echo "请运行一次 !ls \"$GDRIVE_PATH\" 确认文件权限。"
fi


🔎 正在检查云端路径: /content/drive/MyDrive/colab/2026/qwen3_finetune/llama.cpp.tar.gz
✅ 成功找到备份包，启动高速拷贝...
📦 正在解压编译好的环境到根目录...
✨ 环境恢复完成！验证 bin 目录:
/content/llama.cpp/build/bin/llama-server*


In [ ]:
# %%bash
# set -e

# # 如果还没 clone，就 clone
# if [ ! -d "llama.cpp" ]; then
#   git clone https://github.com/ggerganov/llama.cpp
# fi

# cd llama.cpp

# # 创建 build 目录
# mkdir -p build
# cd build

# # 配置（CPU 版本即可，转换不需要 GPU）
# cmake ..

# # 编译
# cmake --build . -j


In [ ]:
# 训练完成后，打包备份到 2T 的 Google Drive 当中
# !tar -czf {gdrive_work_dir}/llama.cpp.tar.gz /content/llama.cpp/


In [ ]:
%%bash
set -e
cd llama.cpp

python convert_hf_to_gguf.py \
  /content/qwen3-emoji-full \
  --outfile /content/qwen3-0.6b-bf16.gguf \
  --outtype bf16


In [ ]:
%%bash
set -e
cd llama.cpp/build/bin

./llama-quantize \
  /content/qwen3-0.6b-bf16.gguf \
  /content/qwen3-0.6b-q4.gguf \
  Q4_K_M


In [ ]:
# 训练完成后，打包备份到 2T 的 Google Drive 当中
# !tar -czf {gdrive_work_dir}/llama.cpp.tar.gz /content/llama.cpp/
!cp -f /content/qwen3-0.6b-bf16.gguf {gdrive_work_dir}/qwen3-0.6b-bf16.gguf
!cp -f /content/qwen3-0.6b-q4.gguf {gdrive_work_dir}/qwen3-0.6b-q4.gguf


In [ ]:
# from google.colab import files
# files.download("/content/qwen3-0.6b-q4.gguf")


In [ ]:
# 1. 安装 cloudflared
!curl -L https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -o cloudflared.deb
!dpkg -i cloudflared.deb

In [ ]:


# 2. 启动你的 llama-server (假设端口是 8080)
# 注意：加 & 让它在后台运行
!/content/llama.cpp/build/bin/llama-server -m /content/qwen3-0.6b-q4.gguf --host 0.0.0.0 --port 18080 > server.log 2>&1 &

# 3. 启动隧道
# 启动隧道并重定向输出，这样你可以看到连接地址
!nohup cloudflared tunnel --url http://localhost:18080 > tunnel.log 2>&1 &

# 给隧道一点启动时间 (3-5秒)
!sleep 5

# 从日志里把那个唯一的 trycloudflared.com 域名扒出来
!grep -o 'https://[-a-z0-9.]*trycloudflared.com' tunnel.log

# 杀掉所有正在运行的隧道，防止端口冲突
# !pkill -9 cloudflared